In [1]:
!pip install decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 25.6 MB/s eta 0:00:00


In [2]:
import os
import json
import time
from typing import List, Dict, Tuple

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor
from decord import VideoReader, cpu

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Based on your earlier screenshots, your data lives under the shared drive
# "DATA 298A" > DATA > MSVD. Adjust this if your actual folder name/path differs.
DATA_ROOT = "/content/drive/Shareddrives/DATA 298A/DATA/MSVD"

print("DATA_ROOT exists:", os.path.exists(DATA_ROOT))
print("msvd_test.json exists:", os.path.exists(os.path.join(DATA_ROOT, "msvd_test.json")))
print("raw_videos folder exists:", os.path.exists(os.path.join(DATA_ROOT, "raw_videos")))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_ROOT exists: True
msvd_test.json exists: True
raw_videos folder exists: True


In [4]:
#Cell 3: helper functions for loading data and building the test set.
def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def sample_frame_indices(total_frames: int, num_frames: int) -> List[int]:
    if total_frames <= 0:
        raise ValueError("Video has no frames")
    return np.linspace(0, total_frames - 1, num_frames, dtype=int).tolist()


def load_video_frames(video_path: str, num_frames: int) -> List[Image.Image]:
    vr = VideoReader(video_path, ctx=cpu(0))
    idxs = sample_frame_indices(len(vr), num_frames)
    frames = vr.get_batch(idxs).asnumpy()
    return [Image.fromarray(frame).convert("RGB") for frame in frames]


def build_test_set(data_root: str) -> Tuple[List[Dict], List[Dict]]:
    test_file = os.path.join(data_root, "msvd_test.json")
    video_root = os.path.join(data_root, "raw_videos")

    if not os.path.exists(test_file):
        raise FileNotFoundError(f"Missing file: {test_file}")
    if not os.path.exists(video_root):
        raise FileNotFoundError(f"Missing folder: {video_root}")

    records = load_json(test_file)

    videos = {}
    queries = []

    for item in records:
        video_name = item["video"]
        video_id = item.get("video_id", video_name)
        video_path = os.path.join(video_root, video_name)

        if not os.path.exists(video_path):
            continue

        if video_id not in videos:
            videos[video_id] = {
                "video_id": video_id,
                "video_name": video_name,
                "video_path": video_path,
            }

        captions = item["caption"]
        if isinstance(captions, str):
            captions = [captions]

        for cap in captions:
            cap = " ".join(str(cap).strip().split())
            if cap:
                queries.append({
                    "video_id": video_id,
                    "caption": cap
                })

    return queries, list(videos.values())


# Quick sanity check — don't need to encode anything yet, just confirm the
# JSON structure parses and videos are found on disk.
queries, videos = build_test_set(DATA_ROOT)
print(f"Queries: {len(queries)} | Videos found: {len(videos)}")
print("Sample query:", queries[0] if queries else "NONE")
print("Sample video:", videos[0] if videos else "NONE")

Queries: 27763 | Videos found: 670
Sample query: {'video_id': 'fr9H1WLcF1A_256_261', 'caption': 'two young men are playing table tennis'}
Sample video: {'video_id': 'fr9H1WLcF1A_256_261', 'video_name': 'fr9H1WLcF1A_256_261.avi', 'video_path': '/content/drive/Shareddrives/DATA 298A/DATA/MSVD/raw_videos/fr9H1WLcF1A_256_261.avi'}


In [5]:
#Cell 4: the model wrapper class.
def extract_tensor_features(output, kind="image"):
    if torch.is_tensor(output):
        return output
    if hasattr(output, "image_embeds") and output.image_embeds is not None:
        return output.image_embeds
    if hasattr(output, "text_embeds") and output.text_embeds is not None:
        return output.text_embeds
    if hasattr(output, "pooler_output") and output.pooler_output is not None:
        return output.pooler_output
    if hasattr(output, "last_hidden_state") and output.last_hidden_state is not None:
        return output.last_hidden_state[:, 0, :]
    if isinstance(output, (tuple, list)) and len(output) > 0:
        if torch.is_tensor(output[0]):
            return output[0]
    raise TypeError(f"Could not extract tensor features from {type(output)}")


class CLIPVideoTextModel(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)

    def encode_text(self, input_ids, attention_mask):
        try:
            text_features = self.clip.get_text_features(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
        except Exception:
            out = self.clip.text_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            text_features = extract_tensor_features(out, kind="text")

        text_features = extract_tensor_features(text_features, kind="text")
        return F.normalize(text_features, dim=-1)

    def encode_video(self, pixel_values):
        bsz, t, c, h, w = pixel_values.shape
        flat_pixels = pixel_values.reshape(bsz * t, c, h, w)

        try:
            frame_features = self.clip.get_image_features(pixel_values=flat_pixels)
        except Exception:
            out = self.clip.vision_model(pixel_values=flat_pixels)
            frame_features = extract_tensor_features(out, kind="image")

        frame_features = extract_tensor_features(frame_features, kind="image")
        frame_features = frame_features.reshape(bsz, t, -1)
        video_features = frame_features.mean(dim=1)

        return F.normalize(video_features, dim=-1)


# Load the model — stock pretrained weights, no fine-tuned checkpoint.
# This is the key difference from the fine-tuned evaluation notebook.
MODEL_NAME = "openai/clip-vit-base-patch32"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPVideoTextModel(MODEL_NAME).to(device)
model.eval()
print(f"Loaded stock pretrained weights: {MODEL_NAME} (zero-shot, no fine-tuning)")

Using device: cuda


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loaded stock pretrained weights: openai/clip-vit-base-patch32 (zero-shot, no fine-tuning)


In [9]:
#Cell 5: encoding functions.
@torch.no_grad()
def encode_videos(model, processor, videos, device, num_frames, batch_size):
    all_embeds = []
    video_ids = []

    for start in tqdm(range(0, len(videos), batch_size), desc="Encoding videos"):
        batch = videos[start:start + batch_size]

        frames_nested = [load_video_frames(v["video_path"], num_frames) for v in batch]
        flat_images = [img for frames in frames_nested for img in frames]

        image_inputs = processor(images=flat_images, return_tensors="pt")
        pixel_values = image_inputs["pixel_values"]

        pixel_values = pixel_values.reshape(
            len(batch), num_frames, *pixel_values.shape[1:]
        ).to(device)

        video_embeds = model.encode_video(pixel_values)
        all_embeds.append(video_embeds.cpu())
        video_ids.extend([v["video_id"] for v in batch])

    return torch.cat(all_embeds, dim=0), video_ids


@torch.no_grad()
def encode_queries(model, processor, queries, device, max_text_len, batch_size):
    all_embeds = []
    gt_video_ids = []

    for start in tqdm(range(0, len(queries), batch_size), desc="Encoding texts"):
        batch = queries[start:start + batch_size]
        texts = [q["caption"] for q in batch]
        gt_video_ids.extend([q["video_id"] for q in batch])

        text_inputs = processor(
            text=texts,
            padding=True,
            truncation=True,
            max_length=max_text_len,
            return_tensors="pt"
        )

        text_embeds = model.encode_text(
            text_inputs["input_ids"].to(device),
            text_inputs["attention_mask"].to(device)
        )
        all_embeds.append(text_embeds.cpu())

    return torch.cat(all_embeds, dim=0), gt_video_ids


# Config for this run
NUM_FRAMES = 8
TEXT_BATCH_SIZE = 64
VIDEO_BATCH_SIZE = 16
MAX_TEXT_LEN = 32

import time
start_time = time.time()

video_embeds, video_ids = encode_videos(
    model, processor, videos, device, NUM_FRAMES, VIDEO_BATCH_SIZE
)
print(f"Video embeds shape: {video_embeds.shape}")

text_embeds, gt_video_ids = encode_queries(
    model, processor, queries, device, MAX_TEXT_LEN, TEXT_BATCH_SIZE
)
print(f"Text embeds shape: {text_embeds.shape}")

encode_time = time.time() - start_time
print(f"Encoding took {encode_time:.1f} seconds")

Encoding videos: 100%|██████████| 42/42 [03:51<00:00,  5.50s/it]


Video embeds shape: torch.Size([670, 512])


Encoding texts: 100%|██████████| 434/434 [00:18<00:00, 23.81it/s]

Text embeds shape: torch.Size([27763, 512])
Encoding took 249.4 seconds


In [10]:
#Cell 6 (final): compute similarity, retrieval metrics, and save results.
import numpy as np
def compute_metrics(similarity: np.ndarray, gt_video_ids: List[str], video_ids: List[str]) -> Dict[str, float]:
    video_id_to_idx = {vid: i for i, vid in enumerate(video_ids)}
    gt_indices = np.array([video_id_to_idx[v] for v in gt_video_ids], dtype=np.int64)

    sorted_idx = np.argsort(-similarity, axis=1)
    ranks = []

    for i in range(len(gt_indices)):
        rank = int(np.where(sorted_idx[i] == gt_indices[i])[0][0]) + 1
        ranks.append(rank)

    ranks = np.array(ranks)

    return {
        "R@1": float(np.mean(ranks <= 1)),
        "R@5": float(np.mean(ranks <= 5)),
        "R@10": float(np.mean(ranks <= 10)),
        "MRR": float(np.mean(1.0 / ranks)),
        "MeanRank": float(np.mean(ranks)),
        "MedianRank": float(np.median(ranks)),
        "Top1Accuracy": float(np.mean(ranks == 1)),
        "Top5Accuracy": float(np.mean(ranks <= 5)),
    }


similarity = (text_embeds @ video_embeds.T).numpy()
metrics = compute_metrics(similarity, gt_video_ids, video_ids)
metrics["avg_query_latency_sec"] = float(encode_time / max(len(queries), 1))
metrics["model_type"] = "zero_shot"
metrics["model_name"] = MODEL_NAME
metrics["num_queries"] = len(queries)
metrics["num_videos"] = len(videos)

print("\nZero-Shot CLIP Retrieval Metrics")
for k, v in metrics.items():
    print(f"{k}: {v}")

OUTPUT_JSON = "/content/clip_zeroshot_results.json"
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(f"\nSaved to {OUTPUT_JSON}")


Zero-Shot CLIP Retrieval Metrics
R@1: 0.33090804307891797
R@5: 0.59262327558261
R@10: 0.6970068076216547
MRR: 0.452596668310793
MeanRank: 23.68800201707308
MedianRank: 3.0
Top1Accuracy: 0.33090804307891797
Top5Accuracy: 0.59262327558261
avg_query_latency_sec: 0.008983800619302821
model_type: zero_shot
model_name: openai/clip-vit-base-patch32
num_queries: 27763
num_videos: 670

Saved to /content/clip_zeroshot_results.json
